# WDS Phase 2 — Train Glove + Person YOLO

**Input:** the dataset auto-labeled in Phase 1 at `/content/drive/MyDrive/wds_dataset` (9 classes).
**Output:** `best.pt` (YOLOv8n) trained on just `glove` + `person`, ready for `wds-native/models/hands.pt`.

**Why filter to 2 classes:** other Phase 1 classes had near-zero detections (`paint=0`, `tool=0`, etc.). Training on them wastes capacity and dilutes mAP. Glove (1043) and person (3926) are the strong signals.

**Runtime:** ~5 min for filter + ~15-20 min for training = ~25 min total on Colab T4.

## Step 1 · Confirm T4 GPU + install deps

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU.'
print(f'GPU OK: {torch.cuda.get_device_name(0)}')

In [ ]:
!pip install -q ultralytics==8.3.30

## Step 2 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3 · Filter Phase 1 dataset to glove + person only

Reads every label file from Phase 1, keeps only lines for `glove` (class 1) and `person` (class 2), remaps them to class IDs `0` and `1`. Images that lose all their labels are skipped.

In [ ]:
import shutil, yaml
from pathlib import Path
from tqdm.auto import tqdm

INPUT_DATASET  = '/content/drive/MyDrive/wds_dataset'
OUTPUT_DATASET = '/content/drive/MyDrive/wds_glove_yolo'

# Phase 1 used: 0:hand 1:glove 2:person 3:chair 4:paint 5:tray 6:container 7:tool 8:workpiece
KEEP = {1: 0, 2: 1}                    # original_id → new_id
NEW_CLASS_NAMES = ['glove', 'person']

src = Path(INPUT_DATASET)
dst = Path(OUTPUT_DATASET)
for split in ('train', 'valid'):
    (dst / split / 'images').mkdir(parents=True, exist_ok=True)
    (dst / split / 'labels').mkdir(parents=True, exist_ok=True)

stats = {'kept': 0, 'skipped_empty': 0, 'glove_boxes': 0, 'person_boxes': 0}

for split in ('train', 'valid'):
    imgs = list((src / split / 'images').glob('*.jpg'))
    for img_path in tqdm(imgs, desc=split):
        lbl = src / split / 'labels' / (img_path.stem + '.txt')
        if not lbl.exists():
            stats['skipped_empty'] += 1
            continue
        new_lines = []
        for line in lbl.read_text().strip().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            cls = int(parts[0])
            if cls in KEEP:
                new_cls = KEEP[cls]
                new_lines.append(f'{new_cls} ' + ' '.join(parts[1:]))
                if cls == 1: stats['glove_boxes']  += 1
                else:        stats['person_boxes'] += 1
        if not new_lines:
            stats['skipped_empty'] += 1
            continue
        (dst / split / 'labels' / lbl.name).write_text('\n'.join(new_lines))
        out_img = dst / split / 'images' / img_path.name
        if not out_img.exists():
            shutil.copy(img_path, out_img)
        stats['kept'] += 1

data_yaml = dst / 'data.yaml'
data_yaml.write_text(yaml.dump({
    'path':  str(dst),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    len(NEW_CLASS_NAMES),
    'names': NEW_CLASS_NAMES,
}, default_flow_style=False))

print()
for k, v in stats.items():
    print(f'  {k:<14}: {v}')
print(f'\n  data.yaml      : {data_yaml}')

## Step 4 · Train YOLOv8n

50 epochs, batch 16, 640 px, with early stop after 10 epochs of no improvement. ~15-20 min on T4.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # pre-trained on COCO; we fine-tune on our 2 classes

results = model.train(
    data     = str(data_yaml),
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    patience = 10,
    name     = 'wds_glove_yolo',
    project  = '/content/drive/MyDrive/wds_runs',
    verbose  = True,
)

## Step 5 · Validate and report mAP

Aim: `mAP@0.5 ≥ 0.80` on glove. Person should be even higher (it's a strong COCO class).

In [ ]:
metrics = model.val()
print(f'\nOverall:')
print(f'  mAP@0.5       : {metrics.box.map50:.3f}')
print(f'  mAP@0.5:0.95  : {metrics.box.map:.3f}')
print(f'\nPer-class mAP@0.5:')
for i, name in enumerate(NEW_CLASS_NAMES):
    print(f'  {name:<8}: {metrics.box.maps[i]:.3f}')

## Step 6 · Locate best.pt for download

In [ ]:
import glob
from pathlib import Path

best = sorted(glob.glob('/content/drive/MyDrive/wds_runs/wds_glove_yolo*/weights/best.pt'))[-1]
size_mb = Path(best).stat().st_size / 1e6
print(f'best.pt   : {best}')
print(f'size      : {size_mb:.2f} MB')
print()
print('Next step (on your Windows machine):')
print(f'  1. Open Drive in your browser, navigate to wds_runs/wds_glove_yolo*/weights/')
print(f'  2. Right-click best.pt → Download')
print(f'  3. Move/rename to: C:\\Users\\marvelele\\Desktop\\wds-native\\models\\hands.pt')
print(f'  4. Tell me when it\'s placed — I\'ll wire it into vision.py')

## Step 7 · Quick visual sanity check

Runs the trained model on 6 random validation images and shows boxes.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

test = YOLO(best)
valid_imgs = list(Path(OUTPUT_DATASET, 'valid', 'images').glob('*.jpg'))
random.shuffle(valid_imgs)   # different 6 each run

# Per-class colors — works for 2-class (glove + person) or 3-class (hand + glove + person).
PALETTE = [
    (40, 200, 200),    # 0: hand    (cyan)         — only present in 3-class
    (40, 200, 40),     # 1: glove   (green)        — index 0 in 2-class
    (200, 40, 200),    # 2: person  (magenta)      — index 1 in 2-class
]
# Truncate palette to whatever class count the trained model actually has
PALETTE = PALETTE[-len(NEW_CLASS_NAMES):] if len(NEW_CLASS_NAMES) < 3 else PALETTE

# Scan until we have 6 images with at least one detection — avoids picking empty frames
non_empty = []
for img_path in valid_imgs:
    res = test.predict(str(img_path), conf=0.25, verbose=False)[0]
    if res.boxes is not None and len(res.boxes) > 0:
        non_empty.append((img_path, res))
    if len(non_empty) >= 6:
        break

if not non_empty:
    print("No images with detections found at conf=0.25 — model is silent on this set.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    for ax, (img_path, res) in zip(axes.flat, non_empty):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        for b in res.boxes:
            cls = int(b.cls.item())
            x1, y1, x2, y2 = [int(v) for v in b.xyxy[0].tolist()]
            col = PALETTE[cls]
            cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
            cv2.putText(img, NEW_CLASS_NAMES[cls], (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()